In [1]:
import numpy as np
import pandas as pd

In [3]:
from boostie.model   import boostieModel
from boostie.data    import (
    make_regression_data,
    make_classification_data,
    make_count_data,
    train_test_split,
)
from boostie.metrics import (
    rmse, mae, r_squared,
    log_loss, accuracy,
    confusion_matrix, precision_recall,
)

# Helpers

In [5]:
def section(title: str) -> None:
    print(f"\n{'=' * 60}")
    print(f"  {title}")
    print(f"{'=' * 60}")

def subsection(title: str) -> None:
    print(f"\n  --- {title} ---")

# Example 1 — Regression

## 1. Data

In [4]:
X, y = make_regression_data(n_samples=600, n_features=6, noise=0.5, seed=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(f"\n  Train: {X_train.shape}  |  Test: {X_test.shape}")


  Train: (480, 6)  |  Test: (120, 6)


## 2. Train

In [ ]:
model = boostieModel(
        n_estimators  = 100,
        max_depth     = 4,
        learning_rate = 0.1,
        reg_lambda    = 1.0,
        reg_gamma     = 0.0,
        objective     = "regression",
    )

In [7]:
model.fit(X_train, y_train, verbose=True, log_every=20)

  [round   20/100]  train loss: 0.675906
  [round   40/100]  train loss: 0.202902
  [round   60/100]  train loss: 0.125888
  [round   80/100]  train loss: 0.099395
  [round  100/100]  train loss: 0.082540


XGBoostModel(objective='regression', n_estimators=100, max_depth=4, lr=0.1, lambda=1.0, gamma=0.0)

In [8]:
print(f"\n  {model}")


  XGBoostModel(objective='regression', n_estimators=100, max_depth=4, lr=0.1, lambda=1.0, gamma=0.0)


## 3. Evaluate

In [9]:
preds = model.predict(X_test)
print(f"  RMSE : {rmse(y_test, preds):.4f}")
print(f"  MAE  : {mae(y_test, preds):.4f}")
print(f"  R²   : {r_squared(y_test, preds):.4f}")

  RMSE : 0.7565
  MAE  : 0.5729
  R²   : 0.9315


## 4. Feature Importance

In [11]:
importances = model.feature_importances(n_features=X.shape[1])
for i, imp in enumerate(importances):
    bar = "█" * int(imp * 40)
    print(f"  feature {i}  {imp:.3f}  {bar}")

  feature 0  0.274  ██████████
  feature 1  0.324  ████████████
  feature 2  0.135  █████
  feature 3  0.088  ███
  feature 4  0.087  ███
  feature 5  0.093  ███


## 5. Preview Predictions

In [12]:
df = pd.DataFrame({
        "y_true": y_test[:8].round(3),
        "y_pred": preds[:8].round(3),
        "error":  (preds[:8] - y_test[:8]).round(3),
    })
print(df.to_string(index=False))

 y_true  y_pred  error
 -7.858  -6.408  1.450
 -3.016  -2.904  0.113
 -2.930  -3.820 -0.890
  2.704   2.238 -0.467
  1.458   0.687 -0.771
 -0.939   0.388  1.327
 -2.344  -1.810  0.534
  3.757   1.950 -1.807


# Example 2 — Binary classification

## 1. Data

In [6]:
X, y = make_classification_data(n_samples=600, n_features=6, noise=0.1, seed=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(f"\n  Train: {X_train.shape}  "
    f"(positive rate: {y_train.mean():.2%})")
print(f"  Test:  {X_test.shape}  "
    f"(positive rate: {y_test.mean():.2%})")


  Train: (480, 6)  (positive rate: 52.08%)
  Test:  (120, 6)  (positive rate: 55.83%)


## 2. Train

In [7]:
model = boostieModel(
        n_estimators  = 100,
        max_depth     = 3,
        learning_rate = 0.1,
        reg_lambda    = 1.0,
        reg_gamma     = 0.0,
        objective     = "binary",
    )

In [8]:
model.fit(X_train, y_train, verbose=True, log_every=20)

  [round   20/100]  train loss: 0.124268
  [round   40/100]  train loss: 0.103179
  [round   60/100]  train loss: 0.089707
  [round   80/100]  train loss: 0.078018
  [round  100/100]  train loss: 0.068308


XGBoostModel(objective='binary', n_estimators=100, max_depth=3, lr=0.1, lambda=1.0, gamma=0.0)

## 3. Evaluate

In [9]:
probs = model.predict(X_test)        # probabilities
print(f"  Log-loss  : {log_loss(y_test, probs):.4f}")
print(f"  Accuracy  : {accuracy(y_test, probs):.4f}")

  Log-loss  : 0.5398
  Accuracy  : 0.7500


In [10]:
pr = precision_recall(y_test, probs)
print(f"  Precision : {pr['precision']:.4f}")
print(f"  Recall    : {pr['recall']:.4f}")
print(f"  F1        : {pr['f1']:.4f}")

  Precision : 0.7761
  Recall    : 0.7761
  F1        : 0.7761


## 4. Confusion matrix

In [11]:
cm = confusion_matrix(y_test, probs)
print(f"           pred 0   pred 1")
print(f"  actual 0  {cm[0,0]:>5}    {cm[0,1]:>5}")
print(f"  actual 1  {cm[1,0]:>5}    {cm[1,1]:>5}")

           pred 0   pred 1
  actual 0     38       15
  actual 1     15       52


## 5. Probability calibration preview

In [13]:
bins = np.linspace(0, 1, 11)
counts, _ = np.histogram(probs, bins=bins)
for i, c in enumerate(counts):
    lo, hi = bins[i], bins[i+1]
    bar = "▪" * (c // 2)
    print(f"  [{lo:.1f}–{hi:.1f}]  {c:>4}  {bar}")

  [0.0–0.1]    16  ▪▪▪▪▪▪▪▪
  [0.1–0.2]    16  ▪▪▪▪▪▪▪▪
  [0.2–0.3]    11  ▪▪▪▪▪
  [0.3–0.4]     5  ▪▪
  [0.4–0.5]     5  ▪▪
  [0.5–0.6]     9  ▪▪▪▪
  [0.6–0.7]    10  ▪▪▪▪▪
  [0.7–0.8]    13  ▪▪▪▪▪▪
  [0.8–0.9]    11  ▪▪▪▪▪
  [0.9–1.0]    24  ▪▪▪▪▪▪▪▪▪▪▪▪


# Example 3 — Poisson count regression

## 1. Data

In [14]:
X, y = make_count_data(n_samples=600, n_features=5, seed=2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(f"\n  Train: {X_train.shape}  "
    f"(mean count: {y_train.mean():.2f})")
print(f"  Test:  {X_test.shape}   "
    f"(mean count: {y_test.mean():.2f})")


  Train: (480, 5)  (mean count: 3.97)
  Test:  (120, 5)   (mean count: 4.69)


## 2. Train

In [15]:
model = boostieModel(
    n_estimators  = 100,
    max_depth     = 3,
    learning_rate = 0.05,
    reg_lambda    = 1.0,
    objective     = "poisson",
    base_score    = float(np.log(y_train.mean() + 1e-6)),
)
model.fit(X_train, y_train, verbose=True, log_every=20)

  [round   20/100]  train loss: 5.579151
  [round   40/100]  train loss: 3.271901
  [round   60/100]  train loss: 2.648515
  [round   80/100]  train loss: 2.311635
  [round  100/100]  train loss: 2.087563


XGBoostModel(objective='poisson', n_estimators=100, max_depth=3, lr=0.05, lambda=1.0, gamma=0.0)

## 3. Evaluate  (predict returns expected counts via exp link)

In [16]:
preds = model.predict(X_test)
print(f"  RMSE : {rmse(y_test, preds):.4f}")
print(f"  MAE  : {mae(y_test, preds):.4f}")

  RMSE : 3.1839
  MAE  : 1.9724


In [17]:
df = pd.DataFrame({
        "y_true":      y_test[:8].astype(int),
        "y_pred (μ)":  preds[:8].round(2),
    })
print(df.to_string(index=False))

 y_true  y_pred (μ)
      3        3.20
      0        2.30
      8        6.22
      3        3.44
      6        3.78
      0        2.31
      1        1.41
      4        1.51
